<a href="https://colab.research.google.com/github/ruthdg43/ML-Ruth-Dagim-UGR-9664-16/blob/Lab-4/Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, mean_squared_error

clf_df = pd.read_csv("classification_escalation.csv")
reg_df = pd.read_csv("resolution_time_regression.csv")
display(clf_df.head(), reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [6]:
X = clf_df.drop(columns=["ticket_id", "escalated"])   # ticket_id is just an ID, so I drop it
y = clf_df["escalated"]                                # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(y.value_counts(normalize=True))
print(y_train.mean(), y_test.mean())

escalated
0    0.787778
1    0.212222
Name: proportion, dtype: float64
0.2125 0.2111111111111111


In [7]:
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "k-NN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring="f1")
    rows.append([name, scores["test_score"].mean(), scores["test_score"].std()])

cv_table = pd.DataFrame(rows, columns=["Model", "Mean F1", "Std F1"])
cv_table

,Model,Mean F1,Std F1
0,Logistic Regression,0.392651,0.062747
1,k-NN,0.360835,0.050295
2,Decision Tree,0.330666,0.058126


In [9]:
pipe = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}

grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="f1")
grid.fit(X_train, y_train)

print("Best settings:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

Best settings: {'model__C': 0.1, 'model__class_weight': 'balanced'}
Best CV F1: 0.5199932976178999


In [10]:
y_pred = grid.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[117  25]
 [ 11  27]]
              precision    recall  f1-score   support

           0       0.91      0.82      0.87       142
           1       0.52      0.71      0.60        38

    accuracy                           0.80       180
   macro avg       0.72      0.77      0.73       180
weighted avg       0.83      0.80      0.81       180



In [11]:
Xr = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
yr = reg_df["resolution_time_hours"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=42)

print(yr.describe())
print("Rows at exactly 0.5 hours:", (yr == 0.5).sum())

count    800.000000
mean       5.873160
std        4.144974
min        0.500000
25%        2.631319
50%        5.387386
75%        8.345903
max       27.488229
Name: resolution_time_hours, dtype: float64
Rows at exactly 0.5 hours: 72


In [12]:
r_num = Xr.select_dtypes(include="number").columns
r_cat = Xr.select_dtypes(exclude="number").columns

r_preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), r_num),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), r_cat)
])

r_cv = KFold(n_splits=5, shuffle=True, random_state=42)

r_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(random_state=42)
}

rows = []
for name, model in r_models.items():
    pipe = Pipeline([("prep", r_preprocess), ("model", model)])
    scores = cross_validate(pipe, Xr_train, yr_train, cv=r_cv, scoring="neg_mean_absolute_error")
    mae = -scores["test_score"]          # sklearn gives negative MAE, so I flip the sign
    rows.append([name, mae.mean(), mae.std()])

pd.DataFrame(rows, columns=["Model", "Mean CV MAE", "Std CV MAE"])

,Model,Mean CV MAE,Std CV MAE
0,Linear Regression,2.035134,0.171650
1,Ridge,2.035127,0.171404
2,Decision Tree,3.269207,0.143924


In [13]:
r_pipe = Pipeline([("prep", r_preprocess), ("model", Ridge())])

r_grid = GridSearchCV(r_pipe, {"model__alpha": [0.01, 0.1, 1, 10, 100]},
                      cv=r_cv, scoring="neg_mean_absolute_error")
r_grid.fit(Xr_train, yr_train)

best_idx = r_grid.best_index_
print("Best alpha:", r_grid.best_params_)
print("Mean CV MAE:", -r_grid.cv_results_["mean_test_score"][best_idx])
print("Std CV MAE:", r_grid.cv_results_["std_test_score"][best_idx])

Best alpha: {'model__alpha': 0.1}
Mean CV MAE: 2.035118085555478
Std CV MAE: 0.1716274370525326


In [14]:
pred = r_grid.predict(Xr_test)

print("Test MAE :", mean_absolute_error(yr_test, pred))
print("Test RMSE:", mean_squared_error(yr_test, pred) ** 0.5)

Test MAE : 2.230086144180656
Test RMSE: 3.4027561084563143
